In [ ]:
"""#**4_evaluation**

##**Load Test Data + Models**
"""

from pyspark.sql import SparkSession
from pyspark.ml.classification import LogisticRegressionModel

spark = SparkSession.builder.getOrCreate()

test_data = spark.read.parquet("data/test_features")
lr_model = LogisticRegressionModel.load("models/lr_model")

"""##**Predictions**"""

lr_predictions = lr_model.transform(test_data)

"""##**Accuracy**"""

from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(metricName="accuracy")
accuracy = evaluator.evaluate(lr_predictions)

print("Logistic Regression Accuracy:", accuracy)

"""##**Confusion Matrix**"""

lr_predictions.groupBy("label", "prediction").count().show()

"""##**Scalability Timing**"""

import time

start = time.time()
_ = lr_model.transform(test_data).count()
end = time.time()

print("Prediction Time:", end - start)



"""#**Export Data from Spark for Tableau**

##**Export Model Performance Metrics**
"""

from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import pandas as pd

models = {
    "Logistic Regression": lr_model.transform(test_data),
    "Naive Bayes": nb_model.transform(test_data),
    "Random Forest": rf_model.transform(test_data),
    "SVM": svm_model.transform(test_data)
}

results = []

evaluator_acc = MulticlassClassificationEvaluator(metricName="accuracy")
evaluator_f1 = MulticlassClassificationEvaluator(metricName="f1")

for name, preds in models.items():
    results.append({
        "Model": name,
        "Accuracy": evaluator_acc.evaluate(preds),
        "F1 Score": evaluator_f1.evaluate(preds)
    })

results_df = pd.DataFrame(results)
results_df.to_csv("model_performance.csv", index=False)

"""###**Export Confusion Matrix Data**"""

conf_df = lr_predictions.groupBy("label", "prediction").count().toPandas()
conf_df.to_csv("confusion_matrix.csv", index=False)

"""##**Export Business Insights Data**"""

from pyspark.sql.functions import length

business_df = df.withColumn("review_length", length("review_text")) \
                .groupBy("stars") \
                .avg("review_length") \
                .toPandas()

business_df.to_csv("business_insights.csv", index=False)

"""##**Export Scalability Data**"""

import time

scalability_results = []

for partitions in [100, 200, 400]:
    spark.conf.set("spark.sql.shuffle.partitions", partitions)

    start = time.time()
    _ = lr_model.transform(test_data).count()
    end = time.time()

    scalability_results.append({
        "Partitions": partitions,
        "Execution_Time": end - start
    })

pd.DataFrame(scalability_results).to_csv("scalability_results.csv", index=False)